#**Entrenamiento red neuronal para retropié derecho**
Prueba con modelo large en vez de medium para comparar la precisión de las predicciones

In [1]:
# Instalamos la librería oficial de YOLO
!pip install -q ultralytics

# Verificamos que PyTorch esté detectando la GPU correctamente
import torch
if torch.cuda.is_available():
    print(f"¡Excelente! GPU detectada: {torch.cuda.get_device_name(0)}")
else:
    print("ERROR: No se detectó GPU.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 83.4 MB/s eta 0:00:00
¡Excelente! GPU detectada: NVIDIA L4


In [2]:
# Descomprimir archivo de base de datos
!unzip -q /content/dataset_retropie_der.zip -d /content/

In [3]:
import yaml

# Estructura de puntos (pi1, pi2, p3) = 3 keypoints
data = {
    'path': '/content/dataset_retropie_der', # Ruta base absoluta en Colab
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'pierna-d'},
    'kpt_shape': [4, 3] # 4 keypoints, 3 dimensiones (x, y, visibilidad)
}

# Guardamos el archivo
with open('/content/data.yaml', 'w') as f:
    yaml.dump(data, f)

print("Archivo data.yaml creado correctamente.")

Archivo data.yaml creado correctamente.


In [4]:
import os
import glob
from PIL import Image, ImageFile

# Obligamos a PIL a leer los archivos aunque les falte el marcador final
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Buscamos absolutamente todas las imágenes en tu dataset de Colab
dataset_dir = '/content/dataset_retropie_der/images'
imagenes = glob.glob(f"{dataset_dir}/**/*.png", recursive=True) + glob.glob(f"{dataset_dir}/**/*.jpg", recursive=True)

print(f"Iniciando reconstrucción de {len(imagenes)} imágenes...")
reparadas = 0

for img_path in imagenes:
    try:
        # Abrimos la imagen (PIL ignora el error gracias a la variable de arriba)
        img = Image.open(img_path)
        img.load() # Forzamos la decodificación completa en RAM

        # Al guardarla sobre sí misma, Python escribe un archivo 100% sano desde cero
        img.save(img_path)
        reparadas += 1
    except Exception as e:
        print(f"Error en {img_path}: {e}")

print(f"Se reescribieron con éxito {reparadas} imágenes.")

Iniciando reconstrucción de 2105 imágenes...
Se reescribieron con éxito 2105 imágenes.


###**Entrenamiento**
* Modelo: Yolov8l
* Epocas: 50
* Batch: 8
* Mosaic: 0.5
* Dropout: 0.1

In [ ]:
!yolo task=pose mode=train model=yolov8l-pose.pt data=data.yaml epochs=100 imgsz=1024 batch=8 mosaic=0.5 dropout=0.1 patience=30 name=modelo_retropie_large_final

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hs